# Novelty Check Pipeline: PubChem, ChEMBL, ZINC & BindingDB



## 1. Setup

In [ ]:
# Core dependencies for the PubChem / ChEMBL / ZINC checks
!pip install rdkit pandas requests tqdm --quiet


Only run the next two cells (Chrome + Selenium install) if you also want to run the **BindingDB** section further down. They are slow (~1-2 min) and unnecessary for the PubChem/ChEMBL/ZINC-only workflow.

In [ ]:
# Optional: dependencies for the BindingDB section only
!pip install selenium webdriver-manager beautifulsoup4 --quiet


In [ ]:
# Optional: install headless Google Chrome for the BindingDB section only (Debian/Ubuntu/Colab)
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add - > /dev/null 2>&1
!echo 'deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main' | tee /etc/apt/sources.list.d/google-chrome.list > /dev/null
!apt-get update -qq
!apt-get install -y -qq google-chrome-stable
!apt-get install -y -qq xvfb libglib2.0-0 libnss3 libfontconfig1 \
                    libgconf-2-4 libappindicator1 libdbus-glib-1-2 \
                    libappindicator3-1 libindicator7 xauth \
                    fonts-liberation


In [ ]:
import time
import json
import pandas as pd
import requests
from tqdm import tqdm

from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem.inchi import MolToInchi, InchiToInchiKey

RDLogger.DisableLog("rdApp.*")  # silence RDKit's stereo warnings


## 2. Configuration

Edit these values, then run the whole notebook top to bottom.

In [ ]:
# ------------------- EDIT THESE ---------------------------------
INPUT_CSV       = "Top-20.csv"      # path to your input CSV
SMILES_COL      = "SMILES"                    # name of the SMILES column in that CSV

OUTPUT_CSV      = "novelty_check_results.csv" # combined results (all molecules, all sources)
NOVEL_CSV       = "novel_compounds.csv"       # subset with no hits in any database

REQUEST_DELAY   = 0.25   # seconds between PubChem / ChEMBL / ZINC requests
REQUEST_TIMEOUT = 10     # seconds per request
MAX_RETRIES     = 3      # retries for transient network errors (not for clean 404s)

RUN_BINDINGDB   = True   # set False to skip the slow Selenium-based BindingDB check
# ------------------------------------------------------------------


## 3. Load molecules & compute InChIKeys (once)

Canonicalizes each SMILES and derives its InChIKey a single time. Invalid SMILES are dropped and reported. Duplicate InChIKeys are deduplicated for the lookups below, then the results are mapped back onto every original row.

In [ ]:
def canonicalize_and_inchikey(smiles):
    """Return (canonical_smiles, inchikey) or (None, None) if SMILES is invalid."""
    if not isinstance(smiles, str) or not smiles.strip():
        return None, None
    mol = Chem.MolFromSmiles(smiles.strip())
    if mol is None:
        return None, None
    try:
        canonical = Chem.MolToSmiles(mol)
        inchikey = InchiToInchiKey(MolToInchi(mol))
        return canonical, inchikey
    except Exception:
        return None, None


df = pd.read_csv(INPUT_CSV)
if SMILES_COL not in df.columns:
    raise ValueError(f"Column '{SMILES_COL}' not found. Available columns: {list(df.columns)}")

canon_ik = df[SMILES_COL].apply(canonicalize_and_inchikey)
df["canonical_smiles"] = canon_ik.apply(lambda t: t[0])
df["inchikey"] = canon_ik.apply(lambda t: t[1])

n_invalid = df["inchikey"].isna().sum()
if n_invalid:
    print(f"Dropping {n_invalid} row(s) with unparseable SMILES.")
df = df.dropna(subset=["inchikey"]).reset_index(drop=True)

# Unique InChIKeys to actually query — duplicates are mapped back at the end
unique_inchikeys = df["inchikey"].drop_duplicates().tolist()
print(f"{len(df)} valid molecules ({len(unique_inchikeys)} unique) to check.")


## 4. Check PubChem, ChEMBL, and ZINC (by InChIKey)

Each function returns `True`/`False`/`None` (`None` = request failed after retries, treat as unknown rather than "not found").

In [ ]:
session = requests.Session()


def _get_with_retries(url, **kwargs):
    """GET with basic retry/backoff for transient errors. Returns the Response or None."""
    for attempt in range(MAX_RETRIES):
        try:
            resp = session.get(url, timeout=REQUEST_TIMEOUT, **kwargs)
            if resp.status_code == 200:
                return resp
            if resp.status_code == 404:
                return resp  # clean "not found" — no need to retry
            time.sleep(1.5 * (attempt + 1))
        except requests.exceptions.RequestException:
            time.sleep(1.5 * (attempt + 1))
    return None


def check_pubchem(inchikey):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{inchikey}/cids/JSON"
    resp = _get_with_retries(url)
    if resp is None:
        return None
    return resp.status_code == 200


def check_chembl(inchikey):
    url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/{inchikey}?format=json"
    resp = _get_with_retries(url)
    if resp is None:
        return None
    return resp.status_code == 200


ZINC_URLS = [
    "https://zinc15.docking.org/substances.json",
    "https://zinc20.docking.org/substances.json",
]


def check_zinc(inchikey):
    for base_url in ZINC_URLS:
        resp = _get_with_retries(
            base_url,
            params={"inchikey": inchikey, "output_fields": "zinc_id,smiles"},
            headers={"Accept": "application/json"},
        )
        if resp is None:
            continue
        if resp.status_code == 200:
            try:
                data = resp.json()
            except ValueError:
                continue
            zinc_ids = [rec.get("zinc_id") for rec in data if "zinc_id" in rec]
            if zinc_ids:
                return True, ";".join(zinc_ids)
    return False, ""


In [ ]:
pubchem_results, chembl_results = {}, {}
zinc_results, zinc_ids_results = {}, {}

for ik in tqdm(unique_inchikeys, desc="PubChem"):
    pubchem_results[ik] = check_pubchem(ik)
    time.sleep(REQUEST_DELAY)

for ik in tqdm(unique_inchikeys, desc="ChEMBL"):
    chembl_results[ik] = check_chembl(ik)
    time.sleep(REQUEST_DELAY)

for ik in tqdm(unique_inchikeys, desc="ZINC"):
    in_zinc, zinc_id = check_zinc(ik)
    zinc_results[ik] = in_zinc
    zinc_ids_results[ik] = zinc_id
    time.sleep(REQUEST_DELAY)

df["in_pubchem"] = df["inchikey"].map(pubchem_results)
df["in_chembl"] = df["inchikey"].map(chembl_results)
df["in_zinc"] = df["inchikey"].map(zinc_results)
df["zinc_id"] = df["inchikey"].map(zinc_ids_results)

print("PubChem hits:", (df['in_pubchem'] == True).sum())
print("ChEMBL hits: ", (df['in_chembl'] == True).sum())
print("ZINC hits:   ", (df['in_zinc'] == True).sum())


## 5. Check BindingDB (by SMILES, via headless Chrome)

BindingDB doesn't expose a simple REST lookup-by-structure endpoint, so this section drives a headless browser against its structure-search page, the same approach as the original notebook, but with a shared driver, retries, and a guaranteed `driver.quit()`.

Skip this cell if `RUN_BINDINGDB = False` above.

In [ ]:
def check_bindingdb_batch(smiles_list, wait_seconds=5, max_retries=2):
    """Search BindingDB by structure for each SMILES. Returns a dict {smiles: True/False/None}."""
    from urllib.parse import quote
    from selenium import webdriver
    from selenium.webdriver.chrome.service import Service
    from selenium.webdriver.chrome.options import Options
    from webdriver_manager.chrome import ChromeDriverManager

    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.binary_location = "/usr/bin/google-chrome"

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_options,
    )

    results = {}
    try:
        for smiles in tqdm(smiles_list, desc="BindingDB"):
            search_url = (
                "https://www.bindingdb.org/rwd/bind/ByStructure.jsp"
                "?smiles=" + quote(smiles)
            )

            found = None
            for attempt in range(max_retries):
                try:
                    driver.get(search_url)
                    time.sleep(wait_seconds)
                    page = driver.page_source.lower()

                    if "no compounds found" in page:
                        found = False
                    elif "bindingdb" in page:
                        found = True
                    else:
                        found = None
                    break
                except Exception:
                    time.sleep(2 * (attempt + 1))

            results[smiles] = found
    finally:
        driver.quit()

    return results


if RUN_BINDINGDB:
    unique_smiles = df["canonical_smiles"].drop_duplicates().tolist()
    bindingdb_results = check_bindingdb_batch(unique_smiles)
    df["in_bindingdb"] = df["canonical_smiles"].map(bindingdb_results)
    print("BindingDB hits:", (df['in_bindingdb'] == True).sum())
else:
    df["in_bindingdb"] = None
    print("Skipped BindingDB check (RUN_BINDINGDB = False).")


## 6. Combine results & flag novel compounds

A molecule is flagged `is_novel = True` only if it came back `False` (a clean "not found") in **every** source that was actually checked. Sources that errored out (`None`) are excluded from that source's contribution but don't block novelty on their own — check `checked_all_sources` if you want to be strict about only trusting rows where every lookup succeeded.

In [ ]:
source_cols = [c for c in ["in_pubchem", "in_chembl", "in_zinc", "in_bindingdb"] if c in df.columns]

# True if every checked source came back a clean "not found" (False), ignoring sources
# that errored out (None) rather than mislabeling them as novel.
df["is_novel"] = df[source_cols].apply(
    lambda row: all(v is False for v in row if v is not None) and any(v is not None for v in row),
    axis=1,
)
df["checked_all_sources"] = df[source_cols].notna().all(axis=1)

df.to_csv(OUTPUT_CSV, index=False)

novel = df[df["is_novel"]].copy()
novel.to_csv(NOVEL_CSV, index=False)

print(f"{len(df)} molecules checked.")
print(f"{len(novel)} potentially novel compounds -> {NOVEL_CSV}")
print(f"Full results -> {OUTPUT_CSV}")

novel.head()
